# 01 — Problem Definition & Data Audit

## Objective
Establish schema, quality, target prevalence, temporal coverage, leakage risks, and the chronological train/validation/test protocol.

> **Scientific contract:** fraud labels are validation ground truth, never unsupervised predictors. Chronological splits protect against future leakage. The test period is locked until Notebook 11.

### Outputs
Reproducible artifacts are saved to `artifacts/` and audit evidence to `reports/`. Interactive controls are for investigation/exploration; the underlying tables remain reproducible.

## 1. Load and profile

In [2]:
from pathlib import Path
import sys, json, warnings, numpy as np, pandas as pd
!pip install plotly
import plotly.express as px
from IPython.display import display, Markdown
import ipywidgets as widgets
warnings.filterwarnings("ignore")
ROOT=Path.cwd()
if not (ROOT/"data").exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/"src"))
from pipeline_utils import *
ART=ROOT/"artifacts"; REP=ROOT/"reports"; ART.mkdir(exist_ok=True); REP.mkdir(exist_ok=True)
RANDOM_STATE=42
fraud,ipmap=load_data(); display(fraud.head()); display(Markdown(f"**Transactions:** {len(fraud):,} | **IP ranges:** {len(ipmap):,} | **Fraud rate:** {fraud['class'].mean():.2%}"))
# IP-range coverage audit
mapped=map_ip_country(fraud,ipmap); print('IP-country mapped:', (mapped.ip_country!='Unknown').mean()); display(mapped.ip_country.value_counts().head(15).to_frame('transactions'))


,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class
0,22058,2015-02-24 22:55:49+00:00,2015-04-18 02:47:11+00:00,34,QVPSPJUOCKZAR,SEO,Chrome,M,39,7.327584e+08,0
1,333320,2015-06-07 20:39:50+00:00,2015-06-08 01:38:54+00:00,16,EOGFQPIZPYXFZ,Ads,Chrome,F,53,3.503114e+08,0
2,1359,2015-01-01 18:52:44+00:00,2015-01-01 18:52:45+00:00,15,YSSKYOSJHPPLJ,SEO,Opera,M,53,2.621474e+09,1
3,150084,2015-04-28 21:13:25+00:00,2015-05-04 13:54:50+00:00,44,ATGTXKYKUDUQN,SEO,Safari,M,41,3.840542e+09,0
4,221365,2015-07-21 07:09:52+00:00,2015-09-09 18:40:53+00:00,39,NAUITBZFJKHWW,Ads,Safari,M,45,4.155831e+08,0


**Transactions:** 151,112 | **IP ranges:** 138,846 | **Fraud rate:** 9.36%

IP-country mapped: 0.8546376197787072


,transactions
ip_country,
United States,58049
Unknown,21966
China,12038
Japan,7306
United Kingdom,4490
Korea Republic of,4162
Germany,3646
France,3161
Canada,2975


## 2. Quality gates

In [3]:
quality=pd.DataFrame({"dtype":fraud.dtypes.astype(str),"missing":fraud.isna().sum(),"missing_pct":100*fraud.isna().mean(),"unique":fraud.nunique(dropna=True)}); display(quality)
checks={"duplicates":int(fraud.duplicated().sum()),"invalid_purchase_time":int(fraud.purchase_time.isna().sum()),"invalid_signup_time":int(fraud.signup_time.isna().sum()),"purchase_before_signup":int((fraud.purchase_time<fraud.signup_time).sum()),"negative_value":int((fraud.purchase_value<0).sum())}; display(pd.Series(checks).to_frame("count"))

,dtype,missing,missing_pct,unique
user_id,int64,0,0.0,151112
signup_time,"datetime64[ns, UTC]",0,0.0,151112
purchase_time,"datetime64[ns, UTC]",0,0.0,150679
purchase_value,int64,0,0.0,122
device_id,object,0,0.0,137956
source,object,0,0.0,3
browser,object,0,0.0,5
sex,object,0,0.0,2
age,int64,0,0.0,58
ip_address,float64,0,0.0,143512


,count
duplicates,0
invalid_purchase_time,0
invalid_signup_time,0
purchase_before_signup,0
negative_value,0


## 3. Chronological split

In [4]:
train,val,test,bounds=chronological_split(fraud); display(pd.DataFrame({"period":["train","validation","test"],"rows":[len(train),len(val),len(test)],"fraud_rate":[train['class'].mean(),val['class'].mean(),test['class'].mean()],"start":[train.purchase_time.min(),val.purchase_time.min(),test.purchase_time.min()],"end":[train.purchase_time.max(),val.purchase_time.max(),test.purchase_time.max()]})); save_json({"checks":checks,"bounds":bounds},REP/"data_audit.json")

,period,rows,fraud_rate,start,end
0,train,105778,0.114173,2015-01-01 00:00:44+00:00,2015-08-05 11:06:39+00:00
1,validation,22667,0.045882,2015-08-05 11:07:00+00:00,2015-09-14 02:11:33+00:00
2,test,22667,0.045617,2015-09-14 02:14:58+00:00,2015-12-16 02:56:05+00:00


## 4. Interactive schema explorer

In [5]:
cols=widgets.SelectMultiple(options=list(fraud.columns),value=tuple(fraud.columns[:6]),description="Columns"); rows=widgets.IntSlider(value=15,min=5,max=50,step=5,description="Rows"); out=widgets.Output()
def show(*_):
    with out: out.clear_output(); display(fraud.loc[:,list(cols.value)].head(rows.value))
cols.observe(show,'value'); rows.observe(show,'value'); display(widgets.VBox([cols,rows,out])); show()